In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.dim_products (
    product_id STRING,
    product_name STRING,
    category STRING,
    price DOUBLE,
    ingestion_date TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.dim_customers (
    customer_id STRING,
    first_name STRING,
    last_name STRING,
    email STRING,
    signup_date DATE,
    country STRING,
    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN,
    ingestion_date TIMESTAMP
)
USING DELTA;

In [0]:
# Reading silver
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp

# Silver table path
silver_products_path = "/Volumes/ecom/storage/project/silver/products/"

products_silver = spark.read.format("delta").load(silver_products_path)

#reference
gold_products_table = DeltaTable.forName(spark, "gold.dim_products")


# SCD TYPE 1 Merge
gold_products_table.alias("gold").merge(
    products_silver.alias("silver"),
    "gold.product_id = silver.product_id"
).whenMatchedUpdate(set={
    "product_name": col("silver.product_name"),
    "category": col("silver.category"),
    "price": col("silver.price"),
    "ingestion_date": current_timestamp()
}).whenNotMatchedInsert(values={
    "product_id": col("silver.product_id"),
    "product_name": col("silver.product_name"),
    "category": col("silver.category"),
    "price": col("silver.price"),
    "ingestion_date": current_timestamp()
}).execute()

In [0]:
# Reading silver
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp

silver_customers_path = "/Volumes/ecom/storage/project/silver/customers/"

customers_silver = spark.read.format("delta").load(silver_customers_path)

#reference
gold_customers_table = DeltaTable.forName(spark, "gold.dim_customers")


# SCD TYPE 2 
from pyspark.sql.functions import lit, current_timestamp, col

gold_customers_table.alias("gold").merge(
    customers_silver.alias("silver"),
    "gold.customer_id = silver.customer_id AND gold.is_current = True"
).whenMatchedUpdate(
    condition=(
        "gold.first_name <> silver.first_name OR "
        "gold.last_name <> silver.last_name OR "
        "gold.email <> silver.email OR "
        "gold.country <> silver.country"
    ),
    set={
        "is_current": lit(False),
        "effective_to": current_timestamp()
    }
).whenNotMatchedInsert(
    values={
        "customer_id": col("silver.customer_id"),
        "first_name": col("silver.first_name"),
        "last_name": col("silver.last_name"),
        "email": col("silver.email"),
        "signup_date": col("silver.signup_date"),
        "country": col("silver.country"),
        "effective_from": current_timestamp(),
        "effective_to": lit(None),
        "is_current": lit(True),
        "ingestion_date": current_timestamp()
    }
).execute()

In [0]:
%sql

--optimizing & z-ordering

OPTIMIZE gold.dim_products
ZORDER BY (product_id);

OPTIMIZE gold.dim_customers
ZORDER BY (customer_id);

OPTIMIZE gold.dim_products
ZORDER BY (product_id, category);



Testing >>>>

In [0]:
%sql
--SELECT * FROM gold.dim_products;
SELECT * FROM gold.dim_customers;
--SELECT * FROM gold.fact_orders;
